# Libraries

In [1]:
# Корень репозитория (где pyproject.toml) → в sys.path добавляется src/
import sys
from pathlib import Path

for _p in [Path.cwd(), *Path.cwd().parents]:
    _src = _p / "src"
    if (_p / "pyproject.toml").is_file() and _src.is_dir():
        _s = str(_src)
        if _s not in sys.path:
            sys.path.insert(0, _s)
        break

import json
import random

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm

from models.cnn_encoder import RawCNNClassifier
from models.gaf_encoder import GAFClassifier
from models.mtf_encoder import MTFClassifier
from models.raw_stats_concat import RawStatsConcatClassifier
from models.stats_encoder import StatisticalMLPClassifier
from models.stft_encoder import STFTClassifier
from metrics.metrics import compute_classification_metrics
from data import load_dataset

from image_transformation.methods.gaf_transformation import GAF
from image_transformation.methods.mtf_transformation import MTF
from image_transformation.methods.stft_transformation import STFTSpectrogram

from statistical.quantile_extractor import (
    STAT_METHODS_GLOBAL_TORCH,
    STAT_METHODS_TORCH,
    TorchQuantileExtractor,
)

# Data

In [2]:
basic_motions_dataset = load_dataset("BasicMotions")


In [3]:
print(basic_motions_dataset.X_train.shape)
print(basic_motions_dataset.y_train.shape)
print(basic_motions_dataset.X_test.shape)
print(basic_motions_dataset.y_test.shape)


(40, 6, 100)
(40,)
(40, 6, 100)
(40,)


In [4]:
ford_a_dataset = load_dataset("FordA")
print(ford_a_dataset.X_train.shape)
print(ford_a_dataset.y_train.shape)
print(ford_a_dataset.X_test.shape)
print(ford_a_dataset.y_test.shape)


(3601, 1, 500)
(3601,)
(1320, 1, 500)
(1320,)


# Tools

In [5]:
def save_result(result, name):
    with open(f"{name}.json", "w") as f:
        json.dump(result, f)

# Simple CNN Encoder

In [6]:
dataset_name = "FordA"
ds = ford_a_dataset

CLASS_NAMES = {
    "BasicMotions": ["walking", "resting", "running", "badminton"],
    "FordA": ["FordA_-1", "FordA_1"],
}

batch_size = min(32, len(ds.X_train))
channels = ds.X_train.shape[1]
time_steps = ds.X_train.shape[2]
num_classes = int(ds.y_train.max()) + 1
class_names = CLASS_NAMES.get(dataset_name, [f"class_{i}" for i in range(num_classes)])

n_epochs = 100
lr = 1e-3
seed = 42

In [7]:
def set_seed(s: int) -> None:
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)


set_seed(seed)

device = torch.device(
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)

X_train_t = torch.from_numpy(ds.X_train).float()
y_train_t = torch.from_numpy(ds.y_train).long()
X_test_t = torch.from_numpy(ds.X_test).float()
y_test_t = torch.from_numpy(ds.y_test).long()

train_loader = DataLoader(
    TensorDataset(X_train_t, y_train_t),
    batch_size=batch_size,
    shuffle=True,
    drop_last=False,
)
test_loader = DataLoader(
    TensorDataset(X_test_t, y_test_t),
    batch_size=batch_size,
    shuffle=False,
)

model = RawCNNClassifier(
    in_channels=channels,
    num_classes=num_classes,
    d_model=128,
    hidden_channels=(64, 128, 128),
    kernel_size=5,
    encoder_dropout=0.1,
    head_dropout=0.2,
).to(device)

xb, yb = next(iter(train_loader))
with torch.no_grad():
    print("logits shape:", model(xb.to(device)).shape)

logits shape: torch.Size([32, 2])


In [8]:
device

device(type='mps')

In [9]:
def train_raw_cnn(
    model: nn.Module,
    train_loader: DataLoader,
    device: torch.device,
    *,
    n_epochs: int,
    lr: float,
) -> list[float]:
    model.train()
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    epoch_losses: list[float] = []

    for _ in tqdm(range(n_epochs)):
        running = 0.0
        n_seen = 0
        for x_raw, y in train_loader:
            x_raw = x_raw.to(device)
            y = y.to(device)
            optimizer.zero_grad(set_to_none=True)
            logits = model(x_raw)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()
            bs = y.size(0)
            running += loss.item() * bs
            n_seen += bs
        epoch_losses.append(running / max(n_seen, 1))

    return epoch_losses

In [10]:
epoch_losses = train_raw_cnn(
    model,
    train_loader,
    device,
    n_epochs=n_epochs,
    lr=lr,
)
print("last train loss:", epoch_losses[-1])

100%|██████████| 100/100 [01:46<00:00,  1.07s/it]

last train loss: 0.15793118997477254


In [10]:
@torch.no_grad()
def evaluate_model(
    model,
    dataloader,
    device,
    class_names=None,
):
    model.eval()

    all_preds = []
    all_targets = []
    total_loss = 0.0
    total_samples = 0

    criterion = nn.CrossEntropyLoss()

    for batch in dataloader:
        x_raw, y = batch

        x_raw = x_raw.to(device)
        y = y.to(device)

        logits = model(x_raw)
        loss = criterion(logits, y)

        preds = torch.argmax(logits, dim=1)

        bs = y.size(0)
        total_loss += loss.item() * bs
        total_samples += bs

        all_preds.append(preds.cpu().numpy())
        all_targets.append(y.cpu().numpy())

    y_pred = np.concatenate(all_preds)
    y_true = np.concatenate(all_targets)

    metrics = compute_classification_metrics(
        y_true=y_true,
        y_pred=y_pred,
        class_names=class_names,
    )

    metrics["loss"] = total_loss / total_samples

    return metrics

In [12]:
metrics = evaluate_model(
    model=model,
    dataloader=test_loader,
    device=device,
    class_names=class_names,
)

print("Loss:", metrics["loss"])
print("Accuracy:", metrics["accuracy"])
print("Macro F1:", metrics["macro_f1"])
print("Weighted F1:", metrics["weighted_f1"])

print("Confusion matrix:")
print(metrics["confusion_matrix"])

print("Classification report:")
print(metrics["classification_report"])

Loss: 0.1982240783671538
Accuracy: 0.9272727272727272
Macro F1: 0.9271119250688502
Weighted F1: 0.9272208555940572
Confusion matrix:
[[643  38]
 [ 58 581]]
Classification report:
              precision    recall  f1-score   support

    FordA_-1       0.92      0.94      0.93       681
     FordA_1       0.94      0.91      0.92       639

    accuracy                           0.93      1320
   macro avg       0.93      0.93      0.93      1320
weighted avg       0.93      0.93      0.93      1320



In [13]:
encoder_result = {
    "experiment_name": "E1_raw_cnn_baseline",
    "modalities": ["raw"],
    "fusion": None,
    "encoder": "1D CNN",
    "loss": float(metrics["loss"]),
    "accuracy": float(metrics["accuracy"]),
    "macro_f1": float(metrics["macro_f1"]),
    "weighted_f1": float(metrics["weighted_f1"]),
    "confusion_matrix": np.asarray(metrics["confusion_matrix"]).tolist(),
}
save_result(encoder_result, "results/experiments/E1_raw_cnn_baseline_FordA")

# Encoder for statistical features

Статистические признаки (FordA): TorchQuantileExtractor

FordA: форма временного ряда (B, 1, T) с T=500. **TorchQuantileExtractor нужно кормить torch.Tensor**; для этого пайплайна удобно брать **(B, T)**, например x.squeeze(1) при одном канале: при **stride > 1** включается матрица Ганкеля по размерности времени — для данных вида **(B, 1, T)** результат получается неверным, для **`(B, T)`** — ожидаемый.

**Параметры под последующий энкодер / классификацию:**
- **STAT_PARAMS_COMPACT** — только глобальные + статистики по всему ряду (window_size=0): **фиксированный короткий вектор (28 признаков)**, удобная размерность для MLP без «раздувания» входа и без лишнего переобучения на малых обучающих выборках типа UCR.
- **STAT_PARAMS_WINDOWS** — дополнительно скользящие окна: window_size=12 интерпретируется как **~12 % от T** (см. apply_window_for_stat_feature_torch), **stride=50** уменьшает число окон и размерность; при **T=500** типичная форма признаков — **(..., 100)** (**19 глобальных + 9 окон × 9 локальных** при выбранных числах; точное число окон см. вывод следующей ячейки).

Ниже: размерность train/test после извлечения и пример среза вектора для одного образца.

In [11]:
def ford_a_feat_tensor(X: torch.Tensor) -> torch.Tensor:
    """FordA имеет один канал: (B, 1, T) → (B, T) для стабильного окна/Hankel по времени."""
    X = X.float()
    if X.ndim == 3 and X.shape[1] == 1:
        return X.squeeze(1)
    return X


STAT_PARAMS_COMPACT = {
    "window_size": 0,
    "stride": 1,
    "add_global_features": True,
}

STAT_PARAMS_WINDOWS = {
    "window_size": 12,
    "stride": 50,
    "add_global_features": True,
}


def feats_batched(
    extractor: TorchQuantileExtractor,
    X_2d: torch.Tensor,
    batch_size: int = 512,
) -> torch.Tensor:
    chunks: list[torch.Tensor] = []
    for start in range(0, len(X_2d), batch_size):
        chunk = X_2d[start:start + batch_size]
        chunks.append(extractor.generate_features_from_ts(chunk))
    return torch.cat(chunks, dim=0)


X_train_2d = ford_a_feat_tensor(torch.from_numpy(ford_a_dataset.X_train))
X_test_2d = ford_a_feat_tensor(torch.from_numpy(ford_a_dataset.X_test))

configs = (
    ("COMPACT → энкодер по умолчанию", STAT_PARAMS_COMPACT),
    ("WINDOWS → больше локальной информации", STAT_PARAMS_WINDOWS),
)

for label, stat_params in configs:
    extractor = TorchQuantileExtractor(stat_params)
    F_train = feats_batched(extractor, X_train_2d)
    F_test = feats_batched(extractor, X_test_2d)
    print(label)
    print("  параметры:", stat_params)
    print("  признаки train:", tuple(F_train.shape))
    print("  признаки test: ", tuple(F_test.shape))
    print(
        "  пример строки train[0, :12]:",
        F_train[0, :12].detach().tolist(),
    )
    print()


print(
    "Если window_size=0 — порядок в векторе: первые "
    f"{len(STAT_METHODS_GLOBAL_TORCH)} глобальных, затем {len(STAT_METHODS_TORCH)} статистик по всему ряду."
)
print("Глобальные имена:")
print(" ", ", ".join(STAT_METHODS_GLOBAL_TORCH.keys()))
print("По-серии (локальные) имена:")
print(" ", ", ".join(STAT_METHODS_TORCH.keys()))

COMPACT → энкодер по умолчанию
  параметры: {'window_size': 0, 'stride': 1, 'add_global_features': True}
  признаки train: (3601, 28)
  признаки test:  (1320, 28)
  пример строки train[0, :12]: [0.07592615485191345, -0.8642673492431641, 27.0, 2.8302007194724865e-05, 0.8931005597114563, 1.4827055931091309, 0.9980000257492065, 0.07199999690055847, 0.9695836305618286, 8.664726257324219, 4.462869644165039, 18.423076629638672]

WINDOWS → больше локальной информации
  параметры: {'window_size': 12, 'stride': 50, 'add_global_features': True}
  признаки train: (3601, 100)
  признаки test:  (1320, 100)
  пример строки train[0, :12]: [0.07592615485191345, -0.8642673492431641, 27.0, 2.8302007194724865e-05, 0.8931005597114563, 1.4827055931091309, 0.9980000257492065, 0.07199999690055847, 0.9695836305618286, 8.664726257324219, 4.462869644165039, 18.423076629638672]

Если window_size=0 — порядок в векторе: первые 19 глобальных, затем 9 статистик по всему ряду.
Глобальные имена:
  skewness_, kurtosis_

In [12]:
# Experiment 2a: stat features COMPACT → MLP classifier

DROP_STAT_FEATURES = {"shannon_entropy_"}


def get_stat_feature_names(stat_params: dict, num_features: int) -> list[str]:
    names = list(STAT_METHODS_GLOBAL_TORCH.keys()) if stat_params.get("add_global_features", True) else []
    if stat_params.get("window_size", 0) == 0:
        names.extend(STAT_METHODS_TORCH.keys())
    else:
        n_window_features = len(STAT_METHODS_TORCH)
        n_windows = (num_features - len(names)) // n_window_features
        for window_idx in range(n_windows):
            names.extend(f"window_{window_idx}_{name}" for name in STAT_METHODS_TORCH.keys())
    return names


def drop_stat_features(
    F_train: torch.Tensor,
    F_test: torch.Tensor,
    feature_names: list[str],
    drop_features: set[str] = DROP_STAT_FEATURES,
) -> tuple[torch.Tensor, torch.Tensor, list[str], list[str]]:
    drop_idx = [idx for idx, name in enumerate(feature_names) if name in drop_features]
    keep_idx = [idx for idx in range(len(feature_names)) if idx not in drop_idx]

    if not drop_idx:
        return F_train, F_test, feature_names, []

    keep_idx_t = torch.tensor(keep_idx, dtype=torch.long)
    kept_feature_names = [feature_names[idx] for idx in keep_idx]
    dropped_feature_names = [feature_names[idx] for idx in drop_idx]
    return F_train[:, keep_idx_t], F_test[:, keep_idx_t], kept_feature_names, dropped_feature_names


def replace_nonfinite_with_train_means(
    F_train: torch.Tensor,
    F_test: torch.Tensor,
) -> tuple[torch.Tensor, torch.Tensor]:
    F_train = F_train.float()
    F_test = F_test.float()

    finite_train = torch.isfinite(F_train)
    train_with_nan = torch.where(finite_train, F_train, torch.nan)
    col_mean = torch.nanmean(train_with_nan, dim=0, keepdim=True)
    col_mean = torch.nan_to_num(col_mean, nan=0.0, posinf=0.0, neginf=0.0)

    F_train = torch.where(finite_train, F_train, col_mean)
    F_test = torch.where(torch.isfinite(F_test), F_test, col_mean)
    return F_train, F_test


def standardize_from_train(
    F_train: torch.Tensor,
    F_test: torch.Tensor,
    eps: float = 1e-6,
) -> tuple[torch.Tensor, torch.Tensor]:
    F_train, F_test = replace_nonfinite_with_train_means(F_train, F_test)
    mean = F_train.mean(dim=0, keepdim=True)
    std = F_train.std(dim=0, unbiased=False, keepdim=True).clamp_min(eps)
    F_train = (F_train - mean) / std
    F_test = (F_test - mean) / std
    return torch.nan_to_num(F_train), torch.nan_to_num(F_test)


def train_stat_mlp(
    model: nn.Module,
    train_loader: DataLoader,
    device: torch.device,
    *,
    n_epochs: int,
    lr: float,
) -> list[float]:
    model.train()
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    epoch_losses: list[float] = []

    for _ in tqdm(range(n_epochs)):
        running = 0.0
        n_seen = 0
        for x_stat, y in train_loader:
            x_stat = x_stat.to(device)
            y = y.to(device)
            optimizer.zero_grad(set_to_none=True)
            logits = model(x_stat)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()

            bs = y.size(0)
            running += loss.item() * bs
            n_seen += bs
        epoch_losses.append(running / max(n_seen, 1))

    return epoch_losses


def run_stat_mlp_experiment(
    experiment_name: str,
    stat_params: dict,
    result_path: str,
    *,
    hidden_dims: tuple[int, ...] = (128, 64),
    dropout: float = 0.2,
) -> dict:
    set_seed(seed)

    extractor = TorchQuantileExtractor(stat_params)
    F_train = feats_batched(extractor, X_train_2d)
    F_test = feats_batched(extractor, X_test_2d)
    feature_names = get_stat_feature_names(stat_params, F_train.shape[1])
    F_train, F_test, feature_names, dropped_feature_names = drop_stat_features(
        F_train,
        F_test,
        feature_names,
    )
    print("features train/test:", tuple(F_train.shape), tuple(F_test.shape))
    print("dropped features:", dropped_feature_names)
    print(
        "non-finite before scaling train/test:",
        int((~torch.isfinite(F_train)).sum()),
        int((~torch.isfinite(F_test)).sum()),
    )
    F_train, F_test = standardize_from_train(F_train, F_test)
    print(
        "non-finite after scaling train/test:",
        int((~torch.isfinite(F_train)).sum()),
        int((~torch.isfinite(F_test)).sum()),
    )

    y_train_stat = torch.from_numpy(ds.y_train).long()
    y_test_stat = torch.from_numpy(ds.y_test).long()

    stat_train_loader = DataLoader(
        TensorDataset(F_train.float(), y_train_stat),
        batch_size=batch_size,
        shuffle=True,
        drop_last=False,
    )
    stat_test_loader = DataLoader(
        TensorDataset(F_test.float(), y_test_stat),
        batch_size=batch_size,
        shuffle=False,
    )

    model = StatisticalMLPClassifier(
        in_features=F_train.shape[1],
        num_classes=num_classes,
        hidden_dims=hidden_dims,
        dropout=dropout,
    ).to(device)

    xb, _ = next(iter(stat_train_loader))
    with torch.no_grad():
        print("logits shape:", model(xb.to(device)).shape)

    epoch_losses = train_stat_mlp(
        model,
        stat_train_loader,
        device,
        n_epochs=n_epochs,
        lr=lr,
    )
    print("last train loss:", epoch_losses[-1])

    metrics = evaluate_model(
        model=model,
        dataloader=stat_test_loader,
        device=device,
        class_names=class_names,
    )

    print("Loss:", metrics["loss"])
    print("Accuracy:", metrics["accuracy"])
    print("Macro F1:", metrics["macro_f1"])
    print("Weighted F1:", metrics["weighted_f1"])

    print("Confusion matrix:")
    print(metrics["confusion_matrix"])

    print("Classification report:")
    print(metrics["classification_report"])

    result = {
        "experiment_name": experiment_name,
        "modalities": ["statistical"],
        "fusion": None,
        "encoder": "MLP",
        "stat_params": stat_params,
        "dropped_features": dropped_feature_names,
        "num_features": int(F_train.shape[1]),
        "loss": float(metrics["loss"]),
        "accuracy": float(metrics["accuracy"]),
        "macro_f1": float(metrics["macro_f1"]),
        "weighted_f1": float(metrics["weighted_f1"]),
        "confusion_matrix": np.asarray(metrics["confusion_matrix"]).tolist(),
    }
    save_result(result, result_path)
    return result

In [24]:
compact_mlp_result = run_stat_mlp_experiment(
    experiment_name="E2_stat_mlp_compact",
    stat_params=STAT_PARAMS_COMPACT,
    result_path="results/experiments/E2_stat_mlp_compact_FordA",
)

features train/test: (3601, 27) (1320, 27)
dropped features: ['shannon_entropy_']
non-finite before scaling train/test: 0 0
non-finite after scaling train/test: 0 0
logits shape: torch.Size([32, 2])


100%|██████████| 100/100 [00:21<00:00,  4.71it/s]

last train loss: 0.09033867264085264
Loss: 0.2117472679777579
Accuracy: 0.9227272727272727
Macro F1: 0.9227208860236383
Weighted F1: 0.9227432394863588
Confusion matrix:
[[615  66]
 [ 36 603]]
Classification report:
              precision    recall  f1-score   support

    FordA_-1       0.94      0.90      0.92       681
     FordA_1       0.90      0.94      0.92       639

    accuracy                           0.92      1320
   macro avg       0.92      0.92      0.92      1320
weighted avg       0.92      0.92      0.92      1320



In [25]:
# Experiment 2b: stat features WINDOWS → MLP classifier

windows_mlp_result = run_stat_mlp_experiment(
    experiment_name="E2_stat_mlp_windows",
    stat_params=STAT_PARAMS_WINDOWS,
    result_path="results/experiments/E2_stat_mlp_windows_FordA",
)

features train/test: (3601, 99) (1320, 99)
dropped features: ['shannon_entropy_']
non-finite before scaling train/test: 0 0
non-finite after scaling train/test: 0 0
logits shape: torch.Size([32, 2])


100%|██████████| 100/100 [00:21<00:00,  4.66it/s]

last train loss: 0.062103435632500405
Loss: 0.5032892259684476
Accuracy: 0.871969696969697
Macro F1: 0.8716189035812387
Weighted F1: 0.8718324299916045
Confusion matrix:
[[610  71]
 [ 98 541]]
Classification report:
              precision    recall  f1-score   support

    FordA_-1       0.86      0.90      0.88       681
     FordA_1       0.88      0.85      0.86       639

    accuracy                           0.87      1320
   macro avg       0.87      0.87      0.87      1320
weighted avg       0.87      0.87      0.87      1320



# Raw + stats

## concat

In [13]:
# Experiment 3: raw time series + statistical features → concat fusion

class RawStatsTensorDataset(torch.utils.data.Dataset):
    def __init__(
        self,
        X_raw: torch.Tensor,
        X_stat: torch.Tensor,
        y: torch.Tensor,
    ):
        self.X_raw = X_raw.float()
        self.X_stat = X_stat.float()
        self.y = y.long()

    def __len__(self) -> int:
        return len(self.y)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        return self.X_raw[idx], self.X_stat[idx], self.y[idx]


def prepare_stat_features_for_experiment(
    stat_params: dict,
) -> tuple[torch.Tensor, torch.Tensor, list[str], list[str]]:
    extractor = TorchQuantileExtractor(stat_params)
    F_train = feats_batched(extractor, X_train_2d)
    F_test = feats_batched(extractor, X_test_2d)
    feature_names = get_stat_feature_names(stat_params, F_train.shape[1])
    F_train, F_test, feature_names, dropped_feature_names = drop_stat_features(
        F_train,
        F_test,
        feature_names,
    )
    F_train, F_test = standardize_from_train(F_train, F_test)
    return F_train.float(), F_test.float(), feature_names, dropped_feature_names


def make_raw_stats_loaders(
    stat_params: dict,
) -> tuple[DataLoader, DataLoader, int, list[str]]:
    F_train, F_test, _feature_names, dropped_feature_names = prepare_stat_features_for_experiment(
        stat_params,
    )

    train_loader = DataLoader(
        RawStatsTensorDataset(X_train_t, F_train, y_train_t),
        batch_size=batch_size,
        shuffle=True,
        drop_last=False,
    )
    test_loader = DataLoader(
        RawStatsTensorDataset(X_test_t, F_test, y_test_t),
        batch_size=batch_size,
        shuffle=False,
    )
    return train_loader, test_loader, F_train.shape[1], dropped_feature_names


def train_raw_stats_concat(
    model: nn.Module,
    train_loader: DataLoader,
    device: torch.device,
    *,
    n_epochs: int,
    lr: float,
) -> list[float]:
    model.train()
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    epoch_losses: list[float] = []

    for _ in tqdm(range(n_epochs)):
        running = 0.0
        n_seen = 0
        for x_raw, x_stat, y in train_loader:
            x_raw = x_raw.to(device)
            x_stat = x_stat.to(device)
            y = y.to(device)
            optimizer.zero_grad(set_to_none=True)
            logits = model(x_raw, x_stat)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()

            bs = y.size(0)
            running += loss.item() * bs
            n_seen += bs
        epoch_losses.append(running / max(n_seen, 1))

    return epoch_losses


@torch.no_grad()
def evaluate_raw_stats_concat(
    model: nn.Module,
    dataloader: DataLoader,
    device: torch.device,
    class_names=None,
) -> dict:
    model.eval()
    criterion = nn.CrossEntropyLoss()

    all_preds = []
    all_targets = []
    total_loss = 0.0
    total_samples = 0

    for x_raw, x_stat, y in dataloader:
        x_raw = x_raw.to(device)
        x_stat = x_stat.to(device)
        y = y.to(device)

        logits = model(x_raw, x_stat)
        loss = criterion(logits, y)
        preds = torch.argmax(logits, dim=1)

        bs = y.size(0)
        total_loss += loss.item() * bs
        total_samples += bs
        all_preds.append(preds.cpu().numpy())
        all_targets.append(y.cpu().numpy())

    y_pred = np.concatenate(all_preds)
    y_true = np.concatenate(all_targets)
    metrics = compute_classification_metrics(
        y_true=y_true,
        y_pred=y_pred,
        class_names=class_names,
    )
    metrics["loss"] = total_loss / total_samples
    return metrics

In [14]:
def run_raw_stats_concat_experiment(
    experiment_name: str,
    stat_params: dict,
    result_path: str,
    *,
    d_model: int = 128,
    raw_hidden_channels: tuple[int, ...] = (64, 128, 128),
    stat_hidden_dims: tuple[int, ...] = (128, 64),
    fusion_hidden_dim: int = 128,
    raw_dropout: float = 0.1,
    stat_dropout: float = 0.2,
    fusion_dropout: float = 0.2,
    head_dropout: float = 0.2,
) -> dict:
    set_seed(seed)

    multimodal_train_loader, multimodal_test_loader, num_stat_features, dropped_features = make_raw_stats_loaders(
        stat_params,
    )
    print("raw train/test:", tuple(X_train_t.shape), tuple(X_test_t.shape))
    print("stat features:", num_stat_features)
    print("dropped features:", dropped_features)

    model = RawStatsConcatClassifier(
        raw_in_channels=channels,
        stat_in_features=num_stat_features,
        num_classes=num_classes,
        d_model=d_model,
        raw_hidden_channels=raw_hidden_channels,
        stat_hidden_dims=stat_hidden_dims,
        fusion_hidden_dim=fusion_hidden_dim,
        raw_dropout=raw_dropout,
        stat_dropout=stat_dropout,
        fusion_dropout=fusion_dropout,
        head_dropout=head_dropout,
    ).to(device)

    xb_raw, xb_stat, _ = next(iter(multimodal_train_loader))
    with torch.no_grad():
        print("logits shape:", model(xb_raw.to(device), xb_stat.to(device)).shape)

    epoch_losses = train_raw_stats_concat(
        model,
        multimodal_train_loader,
        device,
        n_epochs=n_epochs,
        lr=lr,
    )
    print("last train loss:", epoch_losses[-1])

    metrics = evaluate_raw_stats_concat(
        model=model,
        dataloader=multimodal_test_loader,
        device=device,
        class_names=class_names,
    )

    print("Loss:", metrics["loss"])
    print("Accuracy:", metrics["accuracy"])
    print("Macro F1:", metrics["macro_f1"])
    print("Weighted F1:", metrics["weighted_f1"])

    print("Confusion matrix:")
    print(metrics["confusion_matrix"])

    print("Classification report:")
    print(metrics["classification_report"])

    result = {
        "experiment_name": experiment_name,
        "modalities": ["raw", "statistical"],
        "fusion": "concat_mlp",
        "raw_encoder": "1D CNN",
        "stats_encoder": "MLP",
        "stat_params": stat_params,
        "dropped_features": dropped_features,
        "num_stat_features": int(num_stat_features),
        "d_model": int(d_model),
        "loss": float(metrics["loss"]),
        "accuracy": float(metrics["accuracy"]),
        "macro_f1": float(metrics["macro_f1"]),
        "weighted_f1": float(metrics["weighted_f1"]),
        "confusion_matrix": np.asarray(metrics["confusion_matrix"]).tolist(),
    }
    save_result(result, result_path)
    return result

In [15]:
raw_stats_compact_result = run_raw_stats_concat_experiment(
    experiment_name="E3_raw_stats_concat_compact",
    stat_params=STAT_PARAMS_COMPACT,
    result_path="results/experiments/E3_raw_stats_concat_compact_FordA",
)

raw train/test: (3601, 1, 500) (1320, 1, 500)
stat features: 27
dropped features: ['shannon_entropy_']
logits shape: torch.Size([32, 2])


100%|██████████| 100/100 [01:47<00:00,  1.08s/it]


last train loss: 0.08067057239886408
Loss: 0.1249436467780139
Accuracy: 0.9613636363636363
Macro F1: 0.9613263255011291
Weighted F1: 0.9613645463846731
Confusion matrix:
[[655  26]
 [ 25 614]]
Classification report:
              precision    recall  f1-score   support

    FordA_-1       0.96      0.96      0.96       681
     FordA_1       0.96      0.96      0.96       639

    accuracy                           0.96      1320
   macro avg       0.96      0.96      0.96      1320
weighted avg       0.96      0.96      0.96      1320



In [16]:
raw_stats_windows_result = run_raw_stats_concat_experiment(
    experiment_name="E3_raw_stats_concat_windows",
    stat_params=STAT_PARAMS_WINDOWS,
    result_path="results/experiments/E3_raw_stats_concat_windows_FordA",
)

raw train/test: (3601, 1, 500) (1320, 1, 500)
stat features: 99
dropped features: ['shannon_entropy_']
logits shape: torch.Size([32, 2])


100%|██████████| 100/100 [01:46<00:00,  1.07s/it]

last train loss: 0.030384765466443947
Loss: 0.36031625930107003
Accuracy: 0.918939393939394
Macro F1: 0.9189103070143962
Weighted F1: 0.9189591730483924
Confusion matrix:
[[619  62]
 [ 45 594]]
Classification report:
              precision    recall  f1-score   support

    FordA_-1       0.93      0.91      0.92       681
     FordA_1       0.91      0.93      0.92       639

    accuracy                           0.92      1320
   macro avg       0.92      0.92      0.92      1320
weighted avg       0.92      0.92      0.92      1320



# Image transformation

## General info

`ImageTransformation` переводит одномерный ряд FordA из формы `(B, T)` в 2D-представление `(B, H, W)`, после чего для CNN добавляется канал: `(B, 1, H, W)`.

- `GAF` (`Gramian Angular Field`) масштабирует ряд, переводит значения в углы и строит матрицу попарных временных отношений. При `image_size=0.25` и `T=500` ожидаем `H=W=ceil(0.25*T)=125`.
- `MTF` (`Markov Transition Field`) дискретизирует значения в `n_bins`, оценивает матрицу переходов и разворачивает ее обратно во временную 2D-карту. При `image_size=0.25` и `T=500` ожидаем `125×125`.
- `STFT` строит спектрограмму `(frequency_bins, frames)`. При `n_fft=64` получаем `frequency_bins=64//2+1=33`; при `window_size=64`, `hop_length=16`, `center=False`, `T=500` получаем `frames=floor((500-64)/16)+1=28`, то есть `33×28`.

В экспериментах ниже изображения нормализуются только по train split, а метрики считаются на test split.

In [15]:
# ImageTransformation overview: GAF, MTF, STFT

IMAGE_BATCH_SIZE = 64

GAF_PARAMS = {
    "method": "summation",
    "overlapping": True,
    "image_size": 0.25,
}

MTF_PARAMS = {
    "image_size": 0.25,
    "n_bins": 8,
    "strategy": "quantile",
    "overlapping": True,
    "flatten": False,
}

STFT_PARAMS = {
    "window_size": 64,
    "hop_length": 16,
    "n_fft": 64,
    "window_type": "hann",
    "center": False,
    "pad_mode": "reflect",
    "power": 2.0,
    "normalized": False,
}

IMAGE_TRANSFORM_CONFIGS = {
    "GAF": (GAF, GAF_PARAMS),
    "MTF": (MTF, MTF_PARAMS),
    "STFT": (STFTSpectrogram, STFT_PARAMS),
}


def transform_images_batched(
    transformer,
    X_2d: torch.Tensor,
    batch_size: int = IMAGE_BATCH_SIZE,
) -> torch.Tensor:
    images: list[torch.Tensor] = []
    for start in range(0, len(X_2d), batch_size):
        chunk = X_2d[start:start + batch_size].float()
        images.append(transformer.transform(chunk).detach().cpu())
    return torch.cat(images, dim=0)


def to_cnn_images(X_img: torch.Tensor) -> torch.Tensor:
    if X_img.ndim != 3:
        raise ValueError(f"Expected image tensor [batch, height, width], got {tuple(X_img.shape)}")
    return X_img.unsqueeze(1).float()


sample_X = X_train_2d[:8]
print("Raw input for transformations:", tuple(sample_X.shape))

for name, (transformer_cls, params) in IMAGE_TRANSFORM_CONFIGS.items():
    transformer = transformer_cls(params)
    sample_img = transformer.transform(sample_X)
    sample_cnn_img = to_cnn_images(sample_img)
    print(name)
    print("  params:", params)
    print("  transform output [batch, H, W]:", tuple(sample_img.shape))
    print("  CNN input [batch, channels, H, W]:", tuple(sample_cnn_img.shape))

Raw input for transformations: (8, 500)
GAF
  params: {'method': 'summation', 'overlapping': True, 'image_size': 0.25}
  transform output [batch, H, W]: (8, 125, 125)
  CNN input [batch, channels, H, W]: (8, 1, 125, 125)
MTF
  params: {'image_size': 0.25, 'n_bins': 8, 'strategy': 'quantile', 'overlapping': True, 'flatten': False}
  transform output [batch, H, W]: (8, 125, 125)
  CNN input [batch, channels, H, W]: (8, 1, 125, 125)
STFT
  params: {'window_size': 64, 'hop_length': 16, 'n_fft': 64, 'window_type': 'hann', 'center': False, 'pad_mode': 'reflect', 'power': 2.0, 'normalized': False}
  transform output [batch, H, W]: (8, 33, 28)
  CNN input [batch, channels, H, W]: (8, 1, 33, 28)


## GAF

In [16]:
# Experiment 4a: GAF images → CNN encoder → task head

IMAGE_LOG1P_TRANSFORMS = {"STFT"}


def standardize_images_from_train(
    X_train_img: torch.Tensor,
    X_test_img: torch.Tensor,
    eps: float = 1e-6,
) -> tuple[torch.Tensor, torch.Tensor]:
    mean = X_train_img.mean(dim=(0, 2, 3), keepdim=True)
    std = X_train_img.std(dim=(0, 2, 3), unbiased=False, keepdim=True).clamp_min(eps)
    return (X_train_img - mean) / std, (X_test_img - mean) / std


def make_image_loaders(
    transform_name: str,
    transformer,
) -> tuple[DataLoader, DataLoader, tuple[int, ...], tuple[int, ...]]:
    X_train_img = transform_images_batched(transformer, X_train_2d)
    X_test_img = transform_images_batched(transformer, X_test_2d)

    if transform_name in IMAGE_LOG1P_TRANSFORMS:
        X_train_img = torch.log1p(X_train_img.clamp_min(0))
        X_test_img = torch.log1p(X_test_img.clamp_min(0))

    transform_shape = tuple(X_train_img.shape[1:])
    X_train_img = to_cnn_images(X_train_img)
    X_test_img = to_cnn_images(X_test_img)
    cnn_shape = tuple(X_train_img.shape[1:])
    X_train_img, X_test_img = standardize_images_from_train(X_train_img, X_test_img)

    train_loader = DataLoader(
        TensorDataset(X_train_img.float(), y_train_t),
        batch_size=batch_size,
        shuffle=True,
        drop_last=False,
    )
    test_loader = DataLoader(
        TensorDataset(X_test_img.float(), y_test_t),
        batch_size=batch_size,
        shuffle=False,
    )
    return train_loader, test_loader, transform_shape, cnn_shape


def train_image_classifier(
    model: nn.Module,
    train_loader: DataLoader,
    device: torch.device,
    *,
    n_epochs: int,
    lr: float,
) -> list[float]:
    model.train()
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    epoch_losses: list[float] = []

    for _ in tqdm(range(n_epochs)):
        running = 0.0
        n_seen = 0
        for x_img, y in train_loader:
            x_img = x_img.to(device)
            y = y.to(device)
            optimizer.zero_grad(set_to_none=True)
            logits = model(x_img)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()

            bs = y.size(0)
            running += loss.item() * bs
            n_seen += bs
        epoch_losses.append(running / max(n_seen, 1))

    return epoch_losses


def run_image_transform_experiment(
    experiment_name: str,
    transform_name: str,
    transformer_cls,
    transform_params: dict,
    classifier_cls,
    result_path: str,
    *,
    d_model: int = 128,
    hidden_channels: tuple[int, ...] = (32, 64, 128),
    encoder_dropout: float = 0.1,
    head_dropout: float = 0.2,
) -> dict:
    set_seed(seed)

    transformer = transformer_cls(transform_params)
    image_train_loader, image_test_loader, transform_shape, cnn_shape = make_image_loaders(
        transform_name,
        transformer,
    )

    print("transform output [H, W]:", transform_shape)
    print("CNN input [channels, H, W]:", cnn_shape)

    model = classifier_cls(
        in_channels=cnn_shape[0],
        num_classes=num_classes,
        d_model=d_model,
        hidden_channels=hidden_channels,
        encoder_dropout=encoder_dropout,
        head_dropout=head_dropout,
    ).to(device)

    xb, _ = next(iter(image_train_loader))
    with torch.no_grad():
        print("logits shape:", model(xb.to(device)).shape)

    epoch_losses = train_image_classifier(
        model,
        image_train_loader,
        device,
        n_epochs=n_epochs,
        lr=lr,
    )
    print("last train loss:", epoch_losses[-1])

    metrics = evaluate_model(
        model=model,
        dataloader=image_test_loader,
        device=device,
        class_names=class_names,
    )

    print("Loss:", metrics["loss"])
    print("Accuracy:", metrics["accuracy"])
    print("Macro F1:", metrics["macro_f1"])
    print("Weighted F1:", metrics["weighted_f1"])

    print("Confusion matrix:")
    print(metrics["confusion_matrix"])

    print("Classification report:")
    print(metrics["classification_report"])

    result = {
        "experiment_name": experiment_name,
        "modalities": ["image_transformation"],
        "transformation": transform_name,
        "transform_params": transform_params,
        "encoder": "2D CNN",
        "transform_shape": list(transform_shape),
        "cnn_input_shape": list(cnn_shape),
        "loss": float(metrics["loss"]),
        "accuracy": float(metrics["accuracy"]),
        "macro_f1": float(metrics["macro_f1"]),
        "weighted_f1": float(metrics["weighted_f1"]),
        "confusion_matrix": np.asarray(metrics["confusion_matrix"]).tolist(),
    }
    save_result(result, result_path)
    return result

In [17]:
gaf_result = run_image_transform_experiment(
    experiment_name="E4_gaf_cnn",
    transform_name="GAF",
    transformer_cls=GAF,
    transform_params=GAF_PARAMS,
    classifier_cls=GAFClassifier,
    result_path="results/experiments/E4_gaf_cnn_FordA",
)

transform output [H, W]: (125, 125)
CNN input [channels, H, W]: (1, 125, 125)
logits shape: torch.Size([32, 2])


100%|██████████| 100/100 [06:15<00:00,  3.75s/it]


last train loss: 0.0577758951632472
Loss: 0.21107804075335013
Accuracy: 0.9325757575757576
Macro F1: 0.9325754093082437
Weighted F1: 0.9325802850534384
Confusion matrix:
[[617  64]
 [ 25 614]]
Classification report:
              precision    recall  f1-score   support

    FordA_-1       0.96      0.91      0.93       681
     FordA_1       0.91      0.96      0.93       639

    accuracy                           0.93      1320
   macro avg       0.93      0.93      0.93      1320
weighted avg       0.93      0.93      0.93      1320



## MTF

In [18]:
# Experiment 4b: MTF images → CNN encoder → task head

mtf_result = run_image_transform_experiment(
    experiment_name="E4_mtf_cnn",
    transform_name="MTF",
    transformer_cls=MTF,
    transform_params=MTF_PARAMS,
    classifier_cls=MTFClassifier,
    result_path="results/experiments/E4_mtf_cnn_FordA",
)

transform output [H, W]: (125, 125)
CNN input [channels, H, W]: (1, 125, 125)
logits shape: torch.Size([32, 2])


100%|██████████| 100/100 [06:13<00:00,  3.74s/it]


last train loss: 0.05447284320963692
Loss: 0.28339568330257225
Accuracy: 0.9333333333333333
Macro F1: 0.9333074585870091
Weighted F1: 0.9333492562541482
Confusion matrix:
[[629  52]
 [ 36 603]]
Classification report:
              precision    recall  f1-score   support

    FordA_-1       0.95      0.92      0.93       681
     FordA_1       0.92      0.94      0.93       639

    accuracy                           0.93      1320
   macro avg       0.93      0.93      0.93      1320
weighted avg       0.93      0.93      0.93      1320



## STFT

In [19]:
# Experiment 4c: STFT spectrograms → CNN encoder → task head

stft_result = run_image_transform_experiment(
    experiment_name="E4_stft_cnn",
    transform_name="STFT",
    transformer_cls=STFTSpectrogram,
    transform_params=STFT_PARAMS,
    classifier_cls=STFTClassifier,
    result_path="results/experiments/E4_stft_cnn_FordA",
)

transform output [H, W]: (33, 28)
CNN input [channels, H, W]: (1, 33, 28)
logits shape: torch.Size([32, 2])


100%|██████████| 100/100 [00:42<00:00,  2.36it/s]

last train loss: 0.016283076372088374
Loss: 0.28487107440631726
Accuracy: 0.9409090909090909
Macro F1: 0.9408895502645502
Weighted F1: 0.9409237463924963
Confusion matrix:
[[633  48]
 [ 30 609]]
Classification report:
              precision    recall  f1-score   support

    FordA_-1       0.95      0.93      0.94       681
     FordA_1       0.93      0.95      0.94       639

    accuracy                           0.94      1320
   macro avg       0.94      0.94      0.94      1320
weighted avg       0.94      0.94      0.94      1320

